# APIs d'IA dans le cloud : clés et points d'entrée

Un modèle servi par un fournisseur s'appelle par une requête HTTP, authentifiée par une clé d'API. Ce TP fait ce que montrent les slides « APIs sur étagère », « Points d'entrée » et « Clés d'API », sans compte de facturation :

1. Lire une clé dans les secrets de Colab ou de Codespaces
2. Appeler une API en HTTP brut
3. Faire le même appel par son SDK
4. Gérer les erreurs, les quotas et les délais
5. Révoquer et remplacer une clé

L'API principale est celle de Gemini (Google) : son palier gratuit ne demande pas de carte bancaire, elle est ouverte en France et dans l'UE, et dans l'Espace économique européen, la Suisse et le Royaume-Uni, Google applique au quota gratuit les conditions d'usage des données du palier payant (vos requêtes ne servent pas à améliorer ses produits ; [conditions](https://ai.google.dev/gemini-api/terms), octobre 2026).

Le notebook tourne sans GPU, sur Colab comme dans le Codespace. Sans clé, chaque cellule qui appelle une API le dit et explique quoi faire : « Exécuter tout » passe quand même.

## Préparation

**Une clé Gemini** : sur [Google AI Studio](https://aistudio.google.com/apikey), connectez-vous avec un compte Google, puis *Create API key*. Copiez-la, sans la coller dans ce notebook.

**La ranger dans les secrets**, sous le nom `GEMINI_API_KEY` :

- Colab : l'icône de clé (*Secrets*) dans la barre de gauche, *Ajouter un secret*, le nom et la valeur, puis activez *Accès au notebook*.
- Codespaces : sur GitHub, *Settings > Codespaces > Codespaces secrets > New secret*, le nom et la valeur, le dépôt `shuuchuu/labs-codespace` dans *Repository access*, puis arrêtez et redémarrez le Codespace : un secret n'arrive que dans un Codespace (re)démarré après lui, sous la forme d'une variable d'environnement.

## 1. Lire une clé dans les secrets

Écrivez `get_secret(name)`, qui renvoie la valeur du secret `name`, ou `None` s'il n'y en a pas :

- dans Colab, avec `userdata.get` du module `google.colab`, en renvoyant `None` quand le secret n'existe pas, que le notebook n'y a pas accès ou que la demande n'aboutit pas ;
- ailleurs (le Codespace, votre machine), où `google.colab` n'existe pas, dans les variables d'environnement.

N'affichez jamais la valeur d'une clé : une sortie de cellule s'enregistre avec le notebook.

**Documentation utile**

- https://github.com/googlecolab/colabtools/blob/main/google/colab/userdata.py
- https://docs.python.org/3/library/os.html#os.environ
- https://docs.github.com/fr/codespaces/managing-your-codespaces/managing-your-account-specific-secrets-for-github-codespaces

In [ ]:
def get_secret(name):
  """La valeur du secret `name`, ou None s'il n'existe pas."""
  # Votre code ici
  ...

### Conseils supplémentaires

- `from google.colab import userdata` lève `ImportError` hors de Colab : c'est là que se fait le repli, dans un `try:` / `except ImportError:`
- `os.environ.get(name)` : la variable d'environnement, ou `None` si elle n'existe pas
- `userdata.get(name)` lève `userdata.SecretNotFoundError` (pas de secret de ce nom), `userdata.NotebookAccessError` (accès au notebook non activé) ou `userdata.TimeoutException` (pas de réponse de l'interface de Colab en 10 secondes)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import os


def get_secret(name):
  """La valeur du secret `name`, ou None s'il n'existe pas."""
  try:
    from google.colab import userdata
  except ImportError:  # hors de Colab (Codespaces) : une variable d'environnement
    return ...
  try:
    return userdata.get(name)
  except (userdata.SecretNotFoundError, ...):
    return None
```

### Solution

In [ ]:
import os


def get_secret(name):
  """La valeur du secret `name`, ou None s'il n'existe pas."""
  try:
    from google.colab import userdata
  except ImportError:  # hors de Colab (Codespaces) : une variable d'environnement
    return os.environ.get(name)
  try:
    return userdata.get(name)
  except (
    userdata.SecretNotFoundError,
    userdata.NotebookAccessError,
    userdata.TimeoutException,
  ):
    return None

## 2. Appeler l'API en HTTP brut

L'API Gemini a un point d'entrée compatible avec celui d'OpenAI : la requête de la slide « Points d'entrée » y fonctionne telle quelle, avec l'URL, le modèle et la clé ci-dessous. Le modèle est une version fixée (`gemini-3.5-flash-lite`, stable, gratuite dans le palier gratuit), pas un alias comme `gemini-flash-latest`, que Google fait pointer vers chaque nouvelle version.

In [ ]:
import httpx

GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
MODEL = "gemini-3.5-flash-lite"
NO_GEMINI_KEY = (
  "Pas de clé GEMINI_API_KEY : créez-la et rangez-la dans les secrets (voir"
  " « Préparation »), puis relancez la cellule qui la lit."
)

GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
print("GEMINI_API_KEY :", "trouvée" if GEMINI_API_KEY else "absente")

Si la clé est là, envoyez avec `httpx.post` une question de votre choix au modèle `MODEL`, la clé dans l'en-tête `Authorization`, avec un délai maximal, et levez une exception si le code de retour est une erreur. Affichez le texte de la réponse, puis son champ `usage` : les jetons comptés, ce que le fournisseur facture (et ce que compte le quota). Sans clé, affichez `NO_GEMINI_KEY`.

**Documentation utile**

- https://www.python-httpx.org/api/#helper-functions
- https://www.python-httpx.org/api/#response
- https://ai.google.dev/gemini-api/docs/openai

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `httpx.post(url, headers=..., json=..., timeout=60)` : `json=` encode le dictionnaire et pose l'en-tête `Content-Type: application/json` ; sans `timeout`, httpx abandonne après 5 secondes, trop peu pour un LLM
- l'en-tête : `{"Authorization": f"Bearer {GEMINI_API_KEY}"}`
- le corps : `{"model": MODEL, "messages": [{"role": "user", "content": "..."}]}`
- `response.raise_for_status()` lève `httpx.HTTPStatusError` sur un code 4xx ou 5xx
- `response.json()["choices"][0]["message"]["content"]` : le texte ; `response.json()["usage"]` : les jetons de la question (`prompt_tokens`) et de la réponse (`completion_tokens`)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
if GEMINI_API_KEY:
  response = httpx.post(
    GEMINI_URL,
    headers={"Authorization": ...},
    json={
      "model": MODEL,
      "messages": [{"role": "user", "content": ...}],
    },
    timeout=60,
  )
  response.raise_for_status()
  body = response.json()
  print(body["choices"][0]["message"]["content"])
  print(...)
else:
  print(NO_GEMINI_KEY)
```

### Solution

In [ ]:
if GEMINI_API_KEY:
  response = httpx.post(
    GEMINI_URL,
    headers={"Authorization": f"Bearer {GEMINI_API_KEY}"},
    json={
      "model": MODEL,
      "messages": [
        {"role": "user", "content": "En deux phrases : qu'est-ce qu'une clé d'API ?"}
      ],
    },
    timeout=60,
  )
  response.raise_for_status()
  body = response.json()
  print(body["choices"][0]["message"]["content"])
  print(body["usage"])
else:
  print(NO_GEMINI_KEY)

## 3. Le même appel par le SDK

Refaites la même question avec le SDK de Google, `google-genai` : un client créé avec la clé et un délai maximal, puis `generate_content`. Affichez le texte de la réponse et les jetons comptés. Comparez avec la partie 2 : qu'est-ce que le SDK fait pour vous, et qu'est-ce qu'il vous cache ?

**Documentation utile**

- https://ai.google.dev/gemini-api/docs/text-generation
- https://googleapis.github.io/python-genai/

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `genai.Client(api_key=GEMINI_API_KEY, http_options=types.HttpOptions(timeout=60_000))` : le délai du SDK est en **millisecondes** ; sans `api_key`, le client cherche les variables d'environnement `GEMINI_API_KEY` ou `GOOGLE_API_KEY` (pas les secrets de Colab, qui n'en sont pas)
- `client.models.generate_content(model=MODEL, contents="...")` renvoie une réponse typée : `response.text`, `response.usage_metadata`
- le SDK choisit l'URL, pose l'en-tête de la clé (`x-goog-api-key`, celui de l'API native de Gemini), encode et décode le JSON, et transforme une erreur HTTP en exception `google.genai.errors.APIError`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from google import genai
from google.genai import types

if GEMINI_API_KEY:
  client = genai.Client(
    api_key=GEMINI_API_KEY, http_options=types.HttpOptions(timeout=...)
  )
  response = client.models.generate_content(model=MODEL, contents=...)
  print(response.text)
  print(...)
else:
  print(NO_GEMINI_KEY)
```

### Solution

In [ ]:
from google import genai
from google.genai import types

if GEMINI_API_KEY:
  client = genai.Client(
    api_key=GEMINI_API_KEY, http_options=types.HttpOptions(timeout=60_000)
  )
  response = client.models.generate_content(
    model=MODEL, contents="En deux phrases : qu'est-ce qu'une clé d'API ?"
  )
  print(response.text)
  print(response.usage_metadata)
else:
  print(NO_GEMINI_KEY)

## 4. Erreurs, quotas et délais

Une API tombe en erreur, et le code qui l'appelle doit savoir quand réessayer :

| Code | Cause | Que faire |
|---|---|---|
| 400, 401, 403 | Requête mal formée, clé absente, invalide ou révoquée, droits insuffisants | Ne pas réessayer : corriger la requête ou la clé |
| 429 | Quota ou limite de débit atteint (requêtes par minute, par jour…) | Attendre (l'en-tête `Retry-After`, quand il est là, dit combien de secondes), puis réessayer |
| 500, 502, 503, 504 | Panne ou surcharge du fournisseur | Réessayer, en attendant de plus en plus longtemps |
| Délai dépassé | Réseau lent, serveur surchargé | Réessayer, quelques fois au plus |

Les limites du palier gratuit de Gemini dépendent du modèle et du projet : Google AI Studio les affiche ([limites de débit](https://ai.google.dev/gemini-api/docs/rate-limits)).

### 4.1 Réessayer quand il le faut

Écrivez `post_json(client, url, payload, max_attempts=4, base_delay=1.0)`, qui envoie `payload` en JSON avec le client httpx `client` (qui porte déjà la clé et le délai maximal) et renvoie la réponse réussie :

- après un 429, un 5xx ou un délai dépassé, elle attend puis réessaie, `max_attempts` fois en tout au plus ;
- l'attente vaut `Retry-After` quand la réponse le donne, sinon `base_delay`, puis le double à chaque nouvelle tentative (attente exponentielle) ;
- toute autre erreur, et l'erreur de la dernière tentative, lèvent leur exception (`httpx.HTTPStatusError`, `httpx.TimeoutException`).

Affichez chaque nouvel essai et son attente. Les cellules de la partie 4.2 essaient votre fonction.

**Documentation utile**

- https://www.python-httpx.org/api/#client
- https://www.python-httpx.org/exceptions/
- https://developer.mozilla.org/fr/docs/Web/HTTP/Reference/Headers/Retry-After

In [ ]:
def post_json(client, url, payload, max_attempts=4, base_delay=1.0):
  """POST `payload` en JSON, réessaie après un 429, un 5xx ou un délai dépassé.

  Renvoie la réponse réussie ; lève httpx.HTTPStatusError pour une autre erreur
  (requête ou clé refusée) et pour l'erreur de la dernière tentative.
  """
  # Votre code ici
  ...

#### Conseils supplémentaires

- `client.post(url, json=payload)` lève `httpx.TimeoutException` quand le délai est dépassé
- `response.status_code` : le code ; `response.raise_for_status()` lève `httpx.HTTPStatusError` sur une erreur, et renvoie la réponse sinon
- `response.headers.get("Retry-After")` : `None` quand l'en-tête n'y est pas, une chaîne sinon (`"1"`)
- l'attente de la tentative `n` (à partir de 1) : `base_delay * 2 ** (n - 1)`, soit 1, 2, 4 secondes ; `time.sleep(delay)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import time

RETRY_STATUS = {429, 500, 502, 503, 504}


def post_json(client, url, payload, max_attempts=4, base_delay=1.0):
  """POST `payload` en JSON, réessaie après un 429, un 5xx ou un délai dépassé."""
  for attempt in range(1, max_attempts + 1):
    try:
      response = client.post(url, json=payload)
    except httpx.TimeoutException:
      if attempt == max_attempts:
        raise
      delay = ...
      print(f"  tentative {attempt} : délai dépassé, nouvel essai dans {delay:.1f} s")
    else:
      if response.status_code not in RETRY_STATUS or attempt == max_attempts:
        return response.raise_for_status()
      delay = float(response.headers.get("Retry-After", ...))
      print(
        f"  tentative {attempt} : {response.status_code},"
        f" nouvel essai dans {delay:.1f} s"
      )
    time.sleep(delay)
```

#### Solution

In [ ]:
import time

RETRY_STATUS = {429, 500, 502, 503, 504}


def post_json(client, url, payload, max_attempts=4, base_delay=1.0):
  """POST `payload` en JSON, réessaie après un 429, un 5xx ou un délai dépassé.

  Renvoie la réponse réussie ; lève httpx.HTTPStatusError pour une autre erreur
  (requête ou clé refusée) et pour l'erreur de la dernière tentative.
  """
  for attempt in range(1, max_attempts + 1):
    try:
      response = client.post(url, json=payload)
    except httpx.TimeoutException:
      if attempt == max_attempts:
        raise
      delay = base_delay * 2 ** (attempt - 1)
      print(f"  tentative {attempt} : délai dépassé, nouvel essai dans {delay:.1f} s")
    else:
      if response.status_code not in RETRY_STATUS or attempt == max_attempts:
        return response.raise_for_status()
      delay = float(
        response.headers.get("Retry-After", base_delay * 2 ** (attempt - 1))
      )
      print(
        f"  tentative {attempt} : {response.status_code},"
        f" nouvel essai dans {delay:.1f} s"
      )
    time.sleep(delay)

### 4.2 Essayer sur un faux serveur, puis sur le vrai

Provoquer un 429 sur la vraie API consommerait votre quota : `httpx.MockTransport` remplace le réseau par une fonction, qui rejoue ici une suite de réponses (un code HTTP, ou `"timeout"` pour un délai dépassé). Pour chaque scénario, combien de requêtes votre fonction envoie-t-elle, et combien de temps attend-elle ?

In [ ]:
import time


def fake_api(replies):
  """Un client httpx dont le « serveur » rejoue `replies`, une par requête."""
  calls = []

  def handler(request):
    reply = replies[len(calls)]
    calls.append(reply)
    if reply == "timeout":
      raise httpx.ReadTimeout("délai dépassé (simulé)", request=request)
    if reply == 200:
      return httpx.Response(
        200, json={"choices": [{"message": {"content": "Bonjour"}}]}
      )
    headers = {"Retry-After": "1"} if reply == 429 else {}
    return httpx.Response(reply, headers=headers, json={"error": f"erreur {reply}"})

  return httpx.Client(transport=httpx.MockTransport(handler)), calls


scenarios = {
  "quota atteint deux fois": [429, 429, 200],
  "délai dépassé une fois": ["timeout", 200],
  "panne passagère": [503, 503, 200],
  "clé invalide": [401],
  "quota épuisé": [429, 429, 429, 429],
}
for name, replies in scenarios.items():
  print(f"{name} : {replies}")
  client, calls = fake_api(replies)
  start = time.perf_counter()
  try:
    response = post_json(client, "https://api.example/v1/chat", {}, base_delay=0.5)
    outcome = None if response is None else response.status_code
  except httpx.HTTPStatusError as error:
    outcome = f"abandon, {error.response.status_code}"
  except httpx.TimeoutException:
    outcome = "abandon, délai dépassé"
  elapsed = time.perf_counter() - start
  print(f"  → {outcome}, {len(calls)} requête(s) en {elapsed:.1f} s")

Les vraies APIs, avec une clé invalide (cela ne demande pas de compte). Gemini répond `400 INVALID_ARGUMENT` (raison `API_KEY_INVALID`), pas `401` comme la plupart des APIs compatibles avec celle d'OpenAI : lisez le corps de l'erreur, pas seulement son code. Le SDK lève `google.genai.errors.ClientError` (4xx) ou `ServerError` (5xx), deux `APIError` ; il peut réessayer lui-même (`types.HttpOptions(retry_options=types.HttpRetryOptions(attempts=4))`).

In [ ]:
from google import genai
from google.genai import errors

question = {"model": MODEL, "messages": [{"role": "user", "content": "Bonjour"}]}
with httpx.Client(headers={"Authorization": "Bearer cle-invalide"}, timeout=30) as http:
  try:
    post_json(http, GEMINI_URL, question)
  except httpx.HTTPStatusError as error:
    print(error.response.status_code, error.response.text[:200])
  except httpx.TransportError as error:
    print("Pas de réseau :", error)

bad_client = genai.Client(api_key="cle-invalide")
try:
  bad_client.models.generate_content(model=MODEL, contents="Bonjour")
except errors.APIError as error:
  print(type(error).__name__, error.code, error.status, error.message)
except httpx.TransportError as error:
  print("Pas de réseau :", error)

## 5. Révoquer et remplacer une clé

Supposons que `GEMINI_API_KEY` a fuité (collée dans une cellule, poussée sur GitHub). La cellule suivante en garde la valeur actuelle : exécutez-la **avant** de toucher à la clé.

In [ ]:
old_key = GEMINI_API_KEY

Écrivez `key_is_valid(key)`, qui renvoie `True` si Gemini accepte la clé, `False` sinon (en affichant le code de l'erreur). Essayez-la sur une clé inventée, puis faites la rotation :

1. sur la [page des clés de Google AI Studio](https://aistudio.google.com/apikey), créez une nouvelle clé, puis supprimez l'ancienne : elle ne marche plus nulle part ;
2. rangez la nouvelle valeur dans le secret `GEMINI_API_KEY` (Codespaces : puis redémarrez le Codespace, et réexécutez le notebook jusqu'à la cellule précédente) ;
3. relisez le secret, et vérifiez l'ancienne et la nouvelle clé.

**Documentation utile**

- https://ai.google.dev/gemini-api/docs/api-key
- https://googleapis.github.io/python-genai/genai.html#genai.models.Models.get

In [ ]:
def key_is_valid(key):
  """True si Gemini accepte `key`."""
  # Votre code ici
  ...

### Conseils supplémentaires

- `client.models.get(model=MODEL)` lit la fiche du modèle, sans consommer de jetons ; avec une clé refusée, il lève `google.genai.errors.ClientError`, dont `error.code` vaut 400
- gardez le client dans une variable : un `genai.Client(...)` qu'aucune variable ne retient est fermé avant d'envoyer sa requête
- une clé inventée, pour essayer : `"cle-inventee"`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from google import genai
from google.genai import errors


def key_is_valid(key):
  """True si Gemini accepte `key`."""
  client = genai.Client(api_key=...)
  try:
    client.models.get(model=MODEL)
  except errors.ClientError as error:
    print("  refusée :", error.code)
    return False
  return True


print("clé inventée :", key_is_valid("cle-inventee"))
if old_key:
  GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
  print("ancienne clé :", key_is_valid(...))
  print("nouvelle clé :", key_is_valid(...))
```

### Solution

In [ ]:
from google import genai
from google.genai import errors


def key_is_valid(key):
  """True si Gemini accepte `key`."""
  client = genai.Client(api_key=key)
  try:
    client.models.get(model=MODEL)
  except errors.ClientError as error:
    print("  refusée :", error.code)
    return False
  return True


print("clé inventée :", key_is_valid("cle-inventee"))
if old_key:
  GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
  print("ancienne clé :", key_is_valid(old_key))
  print("nouvelle clé :", key_is_valid(GEMINI_API_KEY))
else:
  print(NO_GEMINI_KEY)

## 6. Faire le ménage

Supprimez les clés dont vous n'avez plus besoin, dans Google AI Studio. Une clé par usage, et pas une de plus.